Decision making - 
lists, random module, input function

1. random module - choice
2. Put all recommended anime in a list in python
3. randomly choose one and print

advance idea:

1. make a sublist with anime parameters
2. use input function - mood, runtime, series/movie
3. loop through and find a mood

ADvanced with APIs - jikan module

use pandas to create a table of the anime information

In [ ]:
# version 1 - choosing a random anime to watch from the anime list


from random import choice 

anime = ["naruto", "baki", "shinchan", "death note"]

watch = choice(anime)

print(watch)


naruto


In [18]:
# List of anime. Each anime is a dictionary.
anime = [
    {
        "name": "naruto",
        "genre": "adventure",
        "type": "series"
    },
    {
        "name": "shinchan",
        "genre": "kids",
        "type": "series"
    },
    {
        "name": "death note",
        "genre": "thriller",
        "type": "movie"
    },
    {
        "name": "baki",
        "genre": "action",
        "type": "series"
    }
]

genre = input("Which genre anime do you want to watch?: ").lower()

found = False
for show in anime:
    if genre == show["genre"]:
        print(f"Watch {show['name']}")
        found = True

if not found:
    print("Sorry, no anime found for that genre!")



    




Sorry, no anime found for that genre!


I want to get the list of all animes from jikan api - https://docs.api.jikan.moe/#/anime/getanimefullbyid - and store it in a table using pandas and then based on users preference or mood recommend the anime or animes at a time.

Let me check the Jikan API docs first to understand the available endpoints.

Read URL: https://docs.api.jikan.moe/
Viewed content.md:1-79

The docs page is a Redoc app. Let me fetch the actual API spec to understand the search endpoint.

Read URL: https://api.jikan.moe/v4/genres/anime
Viewed content.md:1-11

Now let me also check the anime search endpoint to understand the response structure:

Read URL: https://api.jikan.moe/v4/anime?genres=1&limit=2&order_by=score&sort=desc
Viewed content.md:1-11

Now I have a clear picture of the API. Here's your step-by-step guide:

---

### Step 1: Install & import what you need

```python
# You may need to install requests first: pip install requests
import requests
import pandas as pd
```

### Step 2: Fetch the list of genres from the API

```python
response = requests.get("https://api.jikan.moe/v4/genres/anime")
genres_data = response.json()["data"]

# Store in a DataFrame so you can see all genres in a table
genres_df = pd.DataFrame(genres_data)[["mal_id", "name"]]
print(genres_df.to_string(index=False))
```

This prints a table like:

```
 mal_id           name
      1         Action
      2      Adventure
      4         Comedy
      8          Drama
     10        Fantasy
     14         Horror
      ...
```

### Step 3: Ask the user to pick a genre

```python
choice = input("What genre are you in the mood for?: ").strip()
```

### Step 4: Find the matching genre ID

The API needs a genre **ID** (like `1` for Action), not the name. So match the user's input:

```python
# Filter genres_df to find the matching genre (case-insensitive)
match = genres_df[genres_df["name"].str.lower() == choice.lower()]

if match.empty:
    print("Genre not found! Please pick from the list above.")
else:
    genre_id = match.iloc[0]["mal_id"]
    print(f"Found genre: {match.iloc[0]['name']} (ID: {genre_id})")
```

### Step 5: Fetch anime for that genre from the API

```python
url = f"https://api.jikan.moe/v4/anime?genres={genre_id}&order_by=score&sort=desc&limit=10"
anime_response = requests.get(url)
anime_list = anime_response.json()["data"]
```

### Step 6: Store results in a pandas table

```python
# Pull out just the fields we care about
anime_rows = []
for a in anime_list:
    anime_rows.append({
        "name": a["title_english"] or a["title"],
        "score": a["score"],
        "episodes": a["episodes"],
        "type": a["type"],
        "status": a["status"],
    })

anime_df = pd.DataFrame(anime_rows)
print("\n🎬 Top anime for your mood:\n")
print(anime_df.to_string(index=False))
```

### Step 7: Put it all together

```python
import requests
import pandas as pd

# Step 1 — Fetch all genres
response = requests.get("https://api.jikan.moe/v4/genres/anime")
genres_data = response.json()["data"]
genres_df = pd.DataFrame(genres_data)[["mal_id", "name"]]

# Step 2 — Show genres to the user
print("Available genres:\n")
print(genres_df["name"].to_string(index=False))

# Step 3 — Ask the user
choice = input("\nWhat genre are you in the mood for?: ").strip()

# Step 4 — Find the genre ID
match = genres_df[genres_df["name"].str.lower() == choice.lower()]

if match.empty:
    print("❌ Genre not found! Please pick from the list above.")
else:
    genre_id = match.iloc[0]["mal_id"]

    # Step 5 — Fetch top anime for that genre
    url = f"https://api.jikan.moe/v4/anime?genres={genre_id}&order_by=score&sort=desc&limit=10"
    anime_response = requests.get(url)
    anime_list = anime_response.json()["data"]

    # Step 6 — Build a table
    anime_rows = []
    for a in anime_list:
        anime_rows.append({
            "Name": a["title_english"] or a["title"],
            "Score": a["score"],
            "Episodes": a["episodes"],
            "Type": a["type"],
            "Status": a["status"],
        })

    anime_df = pd.DataFrame(anime_rows)
    print(f"\n🎬 Top 10 {match.iloc[0]['name']} anime:\n")
    print(anime_df.to_string(index=False))
```

---

**Key concepts you'll practice:**
- **`requests.get()`** — calls the API and gets JSON data back
- **`.json()["data"]`** — parses the JSON response and grabs the `data` list
- **`pd.DataFrame()`** — converts a list of dicts into a table
- **`df[df["col"] == value]`** — filters rows in a DataFrame
- **f-strings in URLs** — dynamically building the API URL with the genre ID

Try building it **one cell at a time** in your notebook — run each step and check the output before moving to the next!

In [29]:
import requests
import pandas as pd

response = requests.get("https://api.jikan.moe/v4/genres/anime") 
#response = stores whatever came back from the URL into a variable called `response`


genres_data = response.json()["data"] 
# Part A — response.json() - The server sent back data as a **JSON string** (basically text). `.json()` converts that text into a **Python dictionary** you can work with.
# Part B — `["data"] - The dictionary has a key called `"data"` which holds the **list** of genres. We only want that list, not the whole dictionary. So `["data"]` grabs just the list.

genres_df = pd.DataFrame(genres_data)[["mal_id", "name"]] # Convert the raw data into a table, then keep only the mal_id and name columns.


print(genres_df)

    mal_id           name
0        1         Action
1        2      Adventure
2        5    Avant Garde
3       46  Award Winning
4       28      Boys Love
..     ...            ...
73      43          Josei
74      15           Kids
75      42         Seinen
76      25         Shoujo
77      27        Shounen

[78 rows x 2 columns]


### Line 1: `response = requests.get("https://api.jikan.moe/v4/genres/anime")`

Think of this like **visiting a website**, but instead of you opening a browser, Python does it for you.

- **`requests.get(...)`** — sends a request to that URL, just like typing it in your browser's address bar and hitting Enter
- **`"https://api.jikan.moe/v4/genres/anime"`** — this is the URL. If you paste this in your browser, you'd see raw data (JSON) instead of a pretty webpage
- **`response = `** — stores whatever came back from the URL into a variable called `response`

`response` now holds the **entire reply** from the server — the status code (200 = success), headers, and the data.


---

### Line 2: `genres_data = response.json()["data"]`

This line does **two things** — let's split it:

**Part A — `response.json()`**

The server sent back data as a **JSON string** (basically text). `.json()` converts that text into a **Python dictionary** you can work with.

It looks something like this:
```python
{
    "data": [
        {"mal_id": 1, "name": "Action"},
        {"mal_id": 2, "name": "Adventure"},
        {"mal_id": 4, "name": "Comedy"},
        ...
    ]
}
```

**Part B — `["data"]`**

The dictionary has a key called `"data"` which holds the **list** of genres. We only want that list, not the whole dictionary. So `["data"]` grabs just the list.

```python
# Without ["data"] → you get the whole dict:
{"data": [{"mal_id": 1, "name": "Action"}, ...]}

# With ["data"] → you get just the list:
[{"mal_id": 1, "name": "Action"}, {"mal_id": 2, "name": "Adventure"}, ...]
```

---

**In plain English:** Line 1 asks the internet for anime genre data. Line 2 converts the reply into a Python list of genres.